<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>RNNs Básicas (Vanilla), GRUs y la función `scan`</h2>
</div>

En este cuaderno, veremos cómo definir el método de propagación hacia adelante (forward) para RNNs básicas (vanilla) y GRUs. Además, veremos cómo definir y usar la función `scan` para calcular la propagación hacia adelante en RNNs.


In [1]:
import numpy as np
from numpy import random
from time import perf_counter

In [2]:
def sigmoid(x): # Funcion sigmoide
    return 1.0 / (1.0 + np.exp(-x))

# Método forward para RNNs básicas (vanilla) y GRUs

Aquí implementaremos el método forward para una RNN básica (vanilla) e implementaremos ese mismo método para una GRU. Usaremos un conjunto de pesos y variables aleatorios con las siguientes dimensiones:

- Tamaño del embedding (`emb`) : 128
- Tamaño del estado oculto (`h_dim`) : (16,1)

Los pesos `w_` y los sesgos `b_` se inicializan con dimensiones (`h_dim`, `emb + h_dim`) y (`h_dim`, 1). El estado oculto `h_t` será un vector columna de tamaño (`h_dim`,1) y el estado oculto inicial `h_0` es un vector de ceros.

In [3]:
random.seed(10)                 # Semilla aleatoria, para que tus resultados coincidan con los nuestros
emb = 128                       # Tamano del embedding
T = 256                         # Numero de variables en las secuencias
h_dim = 16                      # Dimension del estado oculto
h_0 = np.zeros((h_dim, 1))      # Estado oculto inicial
# Inicializacion aleatoria de pesos y sesgos
w1 = random.standard_normal((h_dim, emb+h_dim))
w2 = random.standard_normal((h_dim, emb+h_dim))
w3 = random.standard_normal((h_dim, emb+h_dim))
b1 = random.standard_normal((h_dim, 1))
b2 = random.standard_normal((h_dim, 1))
b3 = random.standard_normal((h_dim, 1))
X = random.standard_normal((T, emb, 1))
weights = [w1, w2, w3, b1, b2, b3]

## Método forward para RNNs básicas (vanilla)

La celda de una RNN básica (vanilla) es bastante sencilla. Su estructura más general se presenta en la siguiente figura: 

<img src="Figures/RNN.PNG" width="400"/>

los cálculos realizados en una celda de RNN básica son equivalentes a las siguientes ecuaciones:

\begin{equation}
h^{<t>}=g(W_{h}[h^{<t-1>},x^{<t>}] + b_h)
\end{equation}
    
\begin{equation}
\hat{y}^{<t>}=g(W_{yh}h^{<t>} + b_y)
\end{equation}

donde $[h^{<t-1>},x^{<t>}]$ significa que $h^{<t-1>}$ y $x^{<t>}$ están concatenados. En la siguiente celda proporcionamos la implementación del método forward para una RNN básica (vanilla). 

In [10]:
def forward_V_RNN(inputs, weights): # Propagacion hacia adelante para una sola celda de RNN basica (vanilla)
    x, h_t = inputs
    # pesos.
    wh, _, _, bh, _, _ = weights
    # nuevo estado oculto
    h_t = np.dot(wh, np.concatenate([h_t, x])) + bh
    h_t = sigmoid(h_t)

    return h_t, h_t

Como podemos ver, omitimos el cálculo de $\hat{y}^{<t>}$. Esto se hizo en aras de la simplicidad, para que podamos concentrarnos en la forma en que se actualizan los estados ocultos aquí y en la celda GRU.

## Método forward para GRUs

Una celda GRU tiene más cálculos que los que tienen las RNNs básicas (vanilla). Esto se puede ver visualmente en el siguiente diagrama:

<img src="Figures/GRU.PNG" width="400"/>

las GRUs tienen compuertas de relevancia $\Gamma_r$ y de actualización $\Gamma_u$ que controlan cómo se actualiza el estado oculto $h^{<t>}$ en cada paso de tiempo. Con estas compuertas, las GRUs son capaces de mantener información relevante en el estado oculto incluso para secuencias largas. Las ecuaciones necesarias para el método forward en las GRUs se proporcionan a continuación: 

\begin{equation}
\Gamma_r=\sigma{(W_r[h^{<t-1>}, x^{<t>}]+b_r)}
\end{equation}

\begin{equation}
\Gamma_u=\sigma{(W_u[h^{<t-1>}, x^{<t>}]+b_u)}
\end{equation}

\begin{equation}
c^{<t>}=\tanh{(W_h[\Gamma_r*h^{<t-1>},x^{<t>}]+b_h)}
\end{equation}

\begin{equation}
h^{<t>}=\Gamma_u*c^{<t>}+(1-\Gamma_u)*h^{<t-1>}
\end{equation}

En la siguiente celda, por favor implementa el método forward para una celda GRU calculando las compuertas de actualización `u` y de relevancia `r`, y el estado oculto candidato `c`. 

In [11]:
def forward_GRU(inputs, weights): # Propagacion hacia adelante para una sola celda GRU
    x, h_t = inputs

    # pesos.
    wu, wr, wc, bu, br, bc = weights

    # Compuerta de actualizacion
    u = np.dot(wu, np.concatenate([h_t, x])) + bu
    u = sigmoid(u)
    
    # Compuerta de relevancia
    r = np.dot(wr, np.concatenate([h_t, x])) + br
    r = sigmoid(u)
    
    # Estado oculto candidato 
    c = np.dot(wc, np.concatenate([r * h_t, x])) + bc
    c = np.tanh(c)
        
    # Nuevo estado oculto h_t
    h_t = u* c + (1 - u)* h_t
    return h_t, h_t

In [12]:
forward_GRU([X[1],h_0], weights)[0]

array([[ 9.77779014e-01],
       [-9.97986240e-01],
       [-5.19958083e-01],
       [-9.99999886e-01],
       [-9.99707004e-01],
       [-3.02197037e-04],
       [-9.58733503e-01],
       [ 2.10804828e-02],
       [ 9.77365398e-05],
       [ 9.99833090e-01],
       [ 1.63200940e-08],
       [ 8.51874303e-01],
       [ 5.21399924e-02],
       [ 2.15495959e-02],
       [ 9.99878828e-01],
       [ 9.77165472e-01]])

# Implementación de la función `scan`

La función `scan` para la propagación hacia adelante en las RNNs, toma como entradas:

- `fn` : la función que se llamará de forma recurrente (es decir, `forward_GRU`)
- `elems` : la lista de entradas para cada paso de tiempo (`X`)
- `weights` : los parámetros necesarios para calcular `fn`
- `h_0` : el estado oculto inicial

`scan` recorre todos los elementos `x` en `elems`, llama a la función `fn` con los argumentos ([`x`, `h_t`],`weights`), almacena el estado oculto calculado `h_t` y agrega el resultado a una lista `ys`. Completa la siguiente celda llamando a `fn` con los argumentos ([`x`, `h_t`],`weights`).

In [13]:
def scan(fn, elems, weights, h_0=None): # Propagacion hacia adelante para RNNs
    h_t = h_0
    ys = []
    for x in elems:
        y, h_t = fn([x, h_t], weights)
        ys.append(y)
    return ys, h_t

# Comparación entre RNNs básicas (vanilla) y GRUs

Ya vimos cómo se calcula la propagación hacia adelante para RNNs básicas (vanilla) y GRUs. Como repaso rápido, necesitamos tener un método forward para la celda recurrente y una función como `scan` para recorrer todos los elementos de una secuencia usando un método forward. Vimos que las GRUs realizaban más cálculos que las RNNs básicas, y podemos comprobar que tienen 3 veces más parámetros. En las siguientes dos celdas, calculamos la propagación hacia adelante para una secuencia con 256 pasos de tiempo (`T`) para una RNN y una GRU con el mismo tamaño de estado oculto `h_t` (`h_dim`=16).  

In [8]:
# RNNs basicas (vanilla)
tic = perf_counter()
ys, h_T = scan(forward_V_RNN, X, weights, h_0)
toc = perf_counter()
RNN_time=(toc-tic)*1000
print (f"La ejecución del método forward para la RNN estándar tomó {RNN_time:.2f} ms.")

It took 1.70ms to run the forward method for the vanilla RNN.


In [9]:
# GRUs
tic = perf_counter()
ys, h_T = scan(forward_GRU, X, weights, h_0)
toc = perf_counter()
GRU_time=(toc-tic)*1000
print (f"La ejecución del método forward de la GRU tomó {GRU_time:.2f} ms.")

It took 4.98ms to run the forward method for the GRU.


Las GRUs toman más tiempo para calcular (Sin embargo, a veces, aunque es algo raro, las RNNs básicas toman más tiempo). Esto significa que el entrenamiento y la predicción tomarían más tiempo para una GRU que para una RNN básica. Sin embargo, las GRUs te permiten propagar información relevante incluso para secuencias largas, así que al seleccionar una arquitectura para PLN deberías evaluar el equilibrio entre el tiempo computacional y el rendimiento. 